In [1]:
# main_yolov8_seg_unified.py
"""
Unified inference: YOLOv8 segmentation model -> vehicles + lane masks -> tracking, speed, curvature, direction.
Usage:
    python main_yolov8_seg_unified.py
Modify CONFIG below (VIDEO_PATH, MODEL_PATH, etc.) before running.
"""

import cv2
import numpy as np
import time, math
from ultralytics import YOLO

# ---------------- CONFIG ----------------
VIDEO_PATH = r"C:\Users\patel\Downloads\curved_lane.mp4"
OUTPUT_PATH = "output_unified_seg.mp4"
MODEL_PATH = "yolov8n-seg.pt"   # use the trained seg checkpoint or yolov8n-seg pretrain if testing
CONF_THRESHOLD = 0.35
DETECT_EVERY_N = 1    # detect every frame (we use model for both boxes + masks); can skip frames if needed
TRACKER_TYPE = "CSRT"
FPS_SMOOTH = 6
PIXELS_TO_METERS_CONST = 25.0
ALLOWED = {"car", "truck", "bus", "motorbike", "motorcycle"}  # match your model names mapping
FORCE_CPU = False
# ----------------------------------------

# ---------- small helpers (same as your original code) ----------
def create_tracker(name="CSRT"):
    return cv2.TrackerCSRT_create() if name == "CSRT" else cv2.TrackerKCF_create()

def euclid(a,b):
    return math.hypot(a[0]-b[0], a[1]-b[1])

def iou(boxA, boxB):
    xx1 = max(boxA[0], boxB[0]); yy1 = max(boxA[1], boxB[1])
    xx2 = min(boxA[2], boxB[2]); yy2 = min(boxA[3], boxB[3])
    interW = max(0, xx2 - xx1); interH = max(0, yy2 - yy1)
    interA = interW * interH
    if interA == 0:
        return 0.0
    aA = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    bA = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])
    return interA / float(aA + bA - interA + 1e-9)

def calc_meters_from_pixels(pixel_distance, frame_h):
    PIXELS_TO_METERS = PIXELS_TO_METERS_CONST / frame_h
    return pixel_distance * PIXELS_TO_METERS

# ---------------- segmentation -> lane polyline conversion ----------------
def get_lane_lines_from_seg(frame, res, lane_class_names=('lane',), min_area=2000):
    """
    Inputs:
      - frame: original frame (H,W,3)
      - res: Ultralytics model result object for this frame (single inference result)
      - lane_class_names: name(s) of classes that indicate lane masks in model.names
    Returns:
      left_pts (list of (x,y)), right_pts (list of (x,y)), curvature_in_meters (float or None)
    """
    h, w = frame.shape[:2]

    # Build binary mask from segmentation result
    mask_full = np.zeros((res.orig_shape[0], res.orig_shape[1]), dtype=np.uint8) if hasattr(res, "orig_shape") else np.zeros((h, w), dtype=np.uint8)

    # res.masks.data shape: (N, H_mask, W_mask) for instance masks (small resized input size)
    try:
        masks = None
        if hasattr(res, "masks") and res.masks is not None:
            # res.masks.data may be on CPU or GPU; ensure numpy array
            try:
                masks = res.masks.data.cpu().numpy()
            except Exception:
                masks = np.array(res.masks.data)
        if masks is None or masks.size == 0:
            mask_small = np.zeros((res.orig_shape[0], res.orig_shape[1]), dtype=np.uint8) if hasattr(res, "orig_shape") else np.zeros((h, w), dtype=np.uint8)
        else:
            # if model provides class indices per mask (res.boxes.cls aligns with masks), we can filter by class name
            # Simpler: union all masks (assuming lane class unique or you've trained with lane masks primarily)
            mask_combined = np.any(masks, axis=0).astype(np.uint8) * 255
            mask_small = mask_combined
    except Exception:
        mask_small = np.zeros((h, w), dtype=np.uint8)

    # Resize mask_small to original frame size if needed
    try:
        small_h, small_w = mask_small.shape
        if (small_w, small_h) != (w, h):
            mask_resized = cv2.resize(mask_small, (w, h), interpolation=cv2.INTER_NEAREST)
        else:
            mask_resized = mask_small
    except Exception:
        mask_resized = cv2.resize(mask_small, (w, h), interpolation=cv2.INTER_NEAREST)

    _, mask_bin = cv2.threshold(mask_resized, 127, 255, cv2.THRESH_BINARY)

    # Connected components -> take largest two components as lanes
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)
    areas = []
    for i in range(1, num_labels):
        areas.append((i, int(stats[i, cv2.CC_STAT_AREA])))
    areas = sorted(areas, key=lambda x: x[1], reverse=True)

    left_pts = []
    right_pts = []
    comp_polylines = []
    for comp_idx, area in areas[:2]:
        if area < min_area:
            continue
        comp_mask = (labels == comp_idx).astype(np.uint8) * 255
        contours, _ = cv2.findContours(comp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not contours:
            continue
        longest = max(contours, key=lambda c: len(c))
        pts = longest.reshape(-1, 2)
        # sort points by y to get polyline top->bottom, sample to limit length
        pts_sorted = pts[np.argsort(pts[:,1])]
        step = max(1, len(pts_sorted) // 120)
        sampled = pts_sorted[::step].tolist()
        comp_polylines.append(sampled)

    # Heuristic assign left/right by bottom-most x
    if len(comp_polylines) == 1:
        left_pts = comp_polylines[0]
        right_pts = []
    elif len(comp_polylines) >= 2:
        bottoms = [poly[-1][0] for poly in comp_polylines]
        if bottoms[0] <= bottoms[1]:
            left_pts = comp_polylines[0]
            right_pts = comp_polylines[1]
        else:
            left_pts = comp_polylines[1]
            right_pts = comp_polylines[0]

    # convert to list of tuples
    left_pts = [(int(x), int(y)) for (x, y) in left_pts] if left_pts else []
    right_pts = [(int(x), int(y)) for (x, y) in right_pts] if right_pts else []

    # curvature computation (same approach as earlier)
    def curvature_from_points(pts, frame_h):
        if not pts or len(pts) < 5:
            return None
        pts_arr = np.array(pts)
        ys = pts_arr[:,1].astype(np.float32)
        xs = pts_arr[:,0].astype(np.float32)
        # fit x = a*y^2 + b*y + c
        try:
            coeff = np.polyfit(ys, xs, 2)
        except Exception:
            return None
        Apx,Bpx,Cpx = coeff
        s = PIXELS_TO_METERS_CONST / frame_h
        ys_m = ys * s
        xs_m = xs * s
        try:
            a,b,c = np.polyfit(ys_m, xs_m, 2)
            y_eval = ys_m[-1]
            dxdy = 2*a*y_eval + b
            d2 = 2*a
            if abs(d2) < 1e-9:
                return None
            R = (1 + dxdy**2)**1.5 / abs(d2)
            return abs(R)
        except Exception:
            return None

    curv_left = curvature_from_points(left_pts, h)
    curv_right = curvature_from_points(right_pts, h)
    curv = None
    if curv_left and curv_right:
        curv = (curv_left + curv_right) / 2.0
    elif curv_left:
        curv = curv_left
    elif curv_right:
        curv = curv_right

    return left_pts, right_pts, curv

# ---------------- MAIN ----------------
def main():
    print("Loading segmentation-capable YOLO model:", MODEL_PATH)
    model = YOLO(MODEL_PATH)
    if FORCE_CPU:
        model.to('cpu')
    else:
        try:
            model.to('cuda')
        except Exception:
            pass
    model.overrides['verbose'] = False

    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        print("Cannot open video:", VIDEO_PATH)
        return
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Video {frame_w}x{frame_h} @ {fps:.2f} FPS")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (frame_w, frame_h))

    trackers = {}
    speed_hist = {}
    next_id = 0
    frame_idx = 0
    lane_state = {"left_fit": None, "right_fit": None, "left_fit_history": [], "right_fit_history": [], "max_history": 6}

    print("Starting processing. press 'q' to quit.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1
        start_t = time.time()

        detections = []
        # Run model every frame (gives both boxes + masks)
        if frame_idx % DETECT_EVERY_N == 0:
            # model inference works better on square seeds - we can resize to 640
            small = cv2.resize(frame, (640, 640))
            res = model(small, imgsz=640, conf=CONF_THRESHOLD, verbose=False)[0]

            # process boxes -> scale back to original frame
            if hasattr(res, "boxes") and len(res.boxes) > 0:
                boxes = res.boxes.xyxy.cpu().numpy()
                classes = res.boxes.cls.cpu().numpy().astype(int)
                confs = res.boxes.conf.cpu().numpy()
                sx = frame.shape[1] / 640.0
                sy = frame.shape[0] / 640.0
                for b,c,cf in zip(boxes, classes, confs):
                    name = model.names[int(c)]
                    if name == "motorcycle":
                        name = "motorbike"
                    if name in ALLOWED:
                        x1,y1,x2,y2 = b[:4]
                        x1 = int(x1 * sx); x2 = int(x2 * sx)
                        y1 = int(y1 * sy); y2 = int(y2 * sy)
                        detections.append((x1,y1,x2,y2,name,float(cf)))

            # Also keep the result object `res` for segmentation mask -> lanes
            res_for_lane = res
        else:
            res_for_lane = None

        # update trackers each frame
        lost = []
        for tid, info in list(trackers.items()):
            ok, box = info['tracker'].update(frame)
            if ok:
                x,y,w_box,h_box = box
                x1 = int(x); y1 = int(y); x2 = int(x + w_box); y2 = int(y + h_box)
                trackers[tid]['bbox'] = (x1,y1,x2,y2)
                trackers[tid]['last_seen'] = frame_idx
            else:
                if frame_idx - info.get('last_seen', frame_idx) > int(fps * 1.0):
                    lost.append(tid)
        for rid in lost:
            trackers.pop(rid, None)
            speed_hist.pop(rid, None)

        # match detection->trackers
        unmatched = []
        for det in detections:
            dx1,dy1,dx2,dy2,dname,dconf = det
            best_iou = 0.0; best_tid = None
            for tid, info in trackers.items():
                val = iou((dx1,dy1,dx2,dy2), info['bbox'])
                if val > best_iou:
                    best_iou = val; best_tid = tid
            if best_iou >= 0.35 and best_tid is not None:
                try:
                    new_tr = create_tracker(TRACKER_TYPE)
                    new_tr.init(frame, (dx1,dy1,dx2-dx1,dy2-dy1))
                    trackers[best_tid]['tracker'] = new_tr
                    trackers[best_tid]['bbox'] = (dx1,dy1,dx2,dy2)
                    trackers[best_tid]['class'] = dname
                    trackers[best_tid]['last_seen'] = frame_idx
                except Exception:
                    pass
            else:
                unmatched.append(det)

        for det in unmatched:
            dx1,dy1,dx2,dy2,dname,dconf = det
            try:
                tr = create_tracker(TRACKER_TYPE)
                tr.init(frame, (dx1,dy1,dx2-dx1,dy2-dy1))
                trackers[next_id] = {'tracker':tr, 'bbox':(dx1,dy1,dx2,dy2), 'positions':[], 'speed_kmh':0.0, 'class':dname, 'last_seen':frame_idx}
                next_id += 1
            except Exception:
                pass

        # get lane lines + curvature from segmentation result (if available)
        left_pts, right_pts, curv = [], [], None
        if res_for_lane is not None:
            left_pts, right_pts, curv = get_lane_lines_from_seg(frame, res_for_lane, lane_class_names=('lane',), min_area=2000)

        # update trackers positions & speeds
        for tid, info in trackers.items():
            x1,y1,x2,y2 = info['bbox']
            if x2 <= x1 or y2 <= y1:
                continue
            bc = ((x1 + x2)//2, y2)
            info['positions'].append((bc, frame_idx))
            if len(info['positions']) > FPS_SMOOTH:
                info['positions'].pop(0)
            if len(info['positions']) >= 2:
                (x_prev, y_prev), f_prev = info['positions'][-2]
                (x_cur, y_cur), f_cur = info['positions'][-1]
                pixel_dist = euclid((x_prev,y_prev),(x_cur,y_cur))
                meters = calc_meters_from_pixels(pixel_dist, frame_h)
                dt_seconds = max(1e-6, (f_cur - f_prev) * (1.0 / fps))
                speed_kmh_new = (meters / dt_seconds) * 3.6
                prev = info.get('speed_kmh', speed_kmh_new)
                info['speed_kmh'] = 0.75 * prev + 0.25 * speed_kmh_new
                if tid not in speed_hist:
                    speed_hist[tid] = []
                speed_hist[tid].append(info['speed_kmh'])
                if len(speed_hist[tid]) > FPS_SMOOTH:
                    speed_hist[tid].pop(0)
                display_speed = float(np.mean(speed_hist[tid]))
            else:
                display_speed = 0.0
            info['display_speed'] = display_speed

        # ------- draw final overlay -------
        vis = frame.copy()

        # Draw lane area polygon if both lanes exist
        if len(left_pts) > 2 and len(right_pts) > 2:
            try:
                lane_polygon = np.array(left_pts + right_pts[::-1], dtype=np.int32)
                cv2.fillPoly(vis, [lane_polygon], (0, 200, 0, 40))
            except Exception:
                pass

        # Draw left and right lane curves
        if len(left_pts) >= 2:
            cv2.polylines(vis, [np.array(left_pts, np.int32)], False, (0, 255, 255), 5)
        if len(right_pts) >= 2:
            cv2.polylines(vis, [np.array(right_pts, np.int32)], False, (0, 255, 255), 5)

        # Draw trackers (boxes, ids, speeds)
        for tid, info in trackers.items():
            x1,y1,x2,y2 = info['bbox']
            clsname = info.get('class','obj')
            spd = info.get('display_speed', 0.0)
            cval = int(min(255, spd * 3))
            color = (0, 255 - cval, cval)
            cv2.rectangle(vis, (x1,y1), (x2,y2), color, 2)
            cv2.putText(vis, f"ID:{tid} {clsname}", (x1, y1-18), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)
            cv2.putText(vis, f"{spd:.1f} km/h", (x1, y2+20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)
            pts = [p[0] for p in info['positions']]
            if len(pts) >= 2:
                cv2.polylines(vis, [np.array(pts, dtype=np.int32)], False, (255,255,0), 2)
            bc = ((x1+x2)//2, y2)
            cv2.circle(vis, bc, 3, (255,255,0), -1)

        # Top-left display: curvature + direction
        if curv is not None:
            display_curv_text = f"Curvature: {curv:.1f} m"
        else:
            display_curv_text = "Curvature: N/A"

        direction = "N/A"
        if len(left_pts) >= 1 and len(right_pts) >= 1:
            left_bottom_x = left_pts[-1][0]
            right_bottom_x = right_pts[-1][0]
            lane_center_x = (left_bottom_x + right_bottom_x) / 2.0
            img_center_x = vis.shape[1] / 2.0
            tol = vis.shape[1] * 0.03  # 3% tolerance
            if lane_center_x - img_center_x > tol:
                direction = "Right"
            elif lane_center_x - img_center_x < -tol:
                direction = "Left"
            else:
                direction = "Straight"

        display_dir_text = f"Direction: {direction}"
        cv2.putText(vis, display_curv_text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (240,240,240), 2, cv2.LINE_AA)
        cv2.putText(vis, display_dir_text,  (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (240,240,240), 2, cv2.LINE_AA)

        # write and show
        out.write(vis)
        cv2.imshow("Unified Lane+Vehicle (seg)", vis)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break

        # optional: print perf
        # print(f"Frame {frame_idx} time: {(time.time()-start_t):.3f}s")

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print("Saved:", OUTPUT_PATH)

if __name__ == "__main__":
    main()


Loading segmentation-capable YOLO model: yolov8n-seg.pt
Video 1280x720 @ 25.00 FPS
Starting processing. press 'q' to quit.
Saved: output_unified_seg.mp4
